In [ ]:
# 1. Google GenAI library நிறுவல்
!pip install -q google-genai

import json
from google import genai
from google.genai import types

# 2. Mock Tools வரையறை (Simulated CBU TAAS Test Scenario)
def get_test_logs() -> str:
    """Returns the test execution logs for the failing CBU TAAS test suite."""
    return """
    [2026-03-30 10:15:00] [INFO] Starting Test: Telecom Billing API Test Suite
    [2026-03-30 10:15:02] [INFO] Sending POST /api/v1/billing/charge
    [2026-03-30 10:15:32] [ERROR] API request timed out after 30 seconds
    [2026-03-30 10:15:32] [FAIL] Test Failed: Telecom Billing API Test
    """

def check_endpoint_status() -> str:
    """Checks the health and status code of the dependent Telecom Billing API endpoint."""
    return json.dumps({
        "endpoint": "https://api.telecom.internal/v1/billing",
        "status_code": 503,
        "message": "Service Unavailable - Database connection pool exhausted"
    })

# Agent-க்கு வழங்கும் Tools பட்டியல்
tools_list = [get_test_logs, check_endpoint_status]

# 3. Gemini Client அமைப்பு (Colab Secrets அல்லது நேரடி Key)
# Colab-ல் API Key-ஐ Secrets-ல் 'GEMINI_API_KEY' என சேமித்து பயன்படுத்தவும்.
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)

# 4. System Instruction (ReAct Loop-க்கான கட்டளை)
system_instruction = """
You are an expert Telecom QA Automation Diagnostic Agent (CBU TAAS).
Your job is to diagnose test failures using available tools and reason step-by-step.

Follow the ReAct framework:
1. Thought: Reason about what to do next.
2. Action: Call an appropriate tool to gather information.
3. Observation: Analyze the tool output.
4. Final Diagnostic Report: Once you have enough information, generate a clear diagnostic report covering:
   - Test Name & Status
   - Root Cause
   - Evidence
   - Recommendation
"""

# 5. Dynamic ReAct Execution Loop (Agent இயங்கும் இடம்)
def run_react_agent():
    print("="*60)
    print("🤖 STARTING REAL REACT AGENT FOR CBU TAAS DIAGNOSTICS")
    print("="*60)

    # Automatic Function Calling வசதியுடன் Chat Session தொடங்கப்படுகிறது
    chat = client.chats.create(
        model="gemini-3.6-flash",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=tools_list,
            temperature=0.2
        )
    )

    # Agent-க்கு ஆரம்பக் கட்டளை வழங்கப்படுகிறது
    prompt = "Diagnose why the recent CBU TAAS test suite failed."
    response = chat.send_message(prompt)

    # Chat history-லிருந்து Agent-ன் சிந்தனை மற்றும் செயல்பாடுகளை அச்சிடுதல்
    for message in chat.get_history():
        role = "Agent" if message.role == "model" else "User / Tool Output"
        print(f"\n--- {role} ---")
        for part in message.parts:
            if part.text:
                print(part.text)
            elif part.function_call:
                print(f"🔧 [ACTION] Calling Tool: {part.function_call.name} with args {part.function_call.args}")
            elif part.function_response:
                print(f"👁️ [OBSERVATION]: {part.function_response.response}")

run_react_agent()

🤖 STARTING REAL REACT AGENT FOR CBU TAAS DIAGNOSTICS

--- User / Tool Output ---
Diagnose why the recent CBU TAAS test suite failed.

--- Agent ---
🔧 [ACTION] Calling Tool: get_test_logs with args {}

--- User / Tool Output ---
👁️ [OBSERVATION]: {'result': '\n    [2026-03-30 10:15:00] [INFO] Starting Test: Telecom Billing API Test Suite\n    [2026-03-30 10:15:02] [INFO] Sending POST /api/v1/billing/charge\n    [2026-03-30 10:15:32] [ERROR] API request timed out after 30 seconds\n    [2026-03-30 10:15:32] [FAIL] Test Failed: Telecom Billing API Test\n    '}

--- Agent ---
🔧 [ACTION] Calling Tool: check_endpoint_status with args {}

--- User / Tool Output ---
👁️ [OBSERVATION]: {'result': '{"endpoint": "https://api.telecom.internal/v1/billing", "status_code": 503, "message": "Service Unavailable - Database connection pool exhausted"}'}

--- Agent ---
### Diagnostic Report: CBU TAAS Test Suite Failure

#### 1. Test Name & Status
* **Test Suite:** Telecom Billing API Test Suite
* **Test Cas